# Fine-tuning DziriBERT — Sentiment Analysis (Derja algérienne)---**Objectif** : Distiller la capacité d'annotation sentiment d'un LLM (Gemini) dans DziriBERT, un modèle BERT pré-entraîné sur le dialecte algérien.**Dataset** : ~1744 commentaires Facebook (Ramy + Hamoud Boualem), annotés par Gemini 2.5 Flash.**Classes** : 3 (positive, negative, neutral) — la classe "mixed" a été fusionnée via soft labels.**Techniques** :- Class weights (inverse frequency) pour gérer le déséquilibre- Focal Loss optionnel (γ=2) pour les cas difficiles- Early stopping sur F1 macro (validation)**Repo** : `rayantr06/ramypulse` → `distillation/`

## 1. Installation & Configuration

In [ ]:
# Installer les dépendances
!pip install -q transformers datasets accelerate scikit-learn tensorboard

# Vérifier le GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Charger le dataset

In [ ]:
# Option A : Depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Option B : Upload direct
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import json
import os

# ── CONFIGURER LE CHEMIN ICI ──
# Si vous avez cloné le repo :
REPO_DIR = "/content/ramypulse"
# Si vous avez uploadé le JSON directement :
# ANNOTATED_JSON = "/content/annotated_comments.json"

# Cloner le repo si nécessaire
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/rayantr06/ramypulse.git {REPO_DIR}

# Exécuter le script de préparation
!pip install -q pandas scikit-learn
!python {REPO_DIR}/distillation/prepare_dataset.py \
    --input /content/annotated_comments.json \
    --output-dir /content/data/

# Vérifier les fichiers générés
!ls -la /content/data/

## 3. Charger la configuration et les splits

In [ ]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

# Charger la config (class weights, label mapping)
with open("/content/data/training_config.json", "r") as f:
    config = json.load(f)

print("Configuration :")
print(json.dumps(config, indent=2, ensure_ascii=False))

# Charger les splits CSV
train_df = pd.read_csv("/content/data/train.csv")
val_df   = pd.read_csv("/content/data/val.csv")
test_df  = pd.read_csv("/content/data/test.csv")

print(f"\nTrain : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
print(f"\nDistribution train :")
print(train_df["label"].value_counts())

# Convertir en HuggingFace Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})
dataset

## 4. Tokenisation avec DziriBERT

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = config["model"]  # "alger-ia/dziribert"
MAX_LENGTH = config["max_seq_length"]  # 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Tokeniser tous les splits
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])

# Renommer label_id → labels (convention HuggingFace)
tokenized = tokenized.rename_column("label_id", "labels")
tokenized.set_format("torch")

print("Colonnes :", tokenized["train"].column_names)
print("Exemple :", tokenized["train"][0])

## 5. Focal Loss + Weighted LossDeux stratégies disponibles — choisir l'une ou l'autre :- **Class Weights** : Pondère la CrossEntropy standard (simple, efficace)- **Focal Loss** : Down-weight les exemples faciles, focus sur les cas difficiles (plus sophistiqué)Par défaut on utilise **Class Weights**. Mettre `USE_FOCAL_LOSS = True` pour switcher.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017) adapté pour la classification multi-classe.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    - gamma=0 : équivalent à CrossEntropy pondérée
    - gamma=2 : recommandé (paper original)
    - alpha : poids par classe (comme class_weights)
    """

    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        targets_one_hot = F.one_hot(targets, num_classes=logits.size(-1)).float()

        # p_t = probabilité de la vraie classe
        p_t = (probs * targets_one_hot).sum(dim=-1)
        focal_weight = (1.0 - p_t) ** self.gamma

        # Log-probabilities pour la stabilité numérique
        log_probs = F.log_softmax(logits, dim=-1)
        ce_loss = -(targets_one_hot * log_probs).sum(dim=-1)

        loss = focal_weight * ce_loss

        # Alpha weighting par classe
        if self.alpha is not None:
            alpha_t = (self.alpha.unsqueeze(0) * targets_one_hot).sum(dim=-1)
            loss = alpha_t * loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ── CHOIX DE LA STRATÉGIE ──
USE_FOCAL_LOSS = False  # Mettre True pour utiliser Focal Loss
FOCAL_GAMMA = 2.0       # Paramètre gamma (2.0 recommandé)

# Préparer les tenseurs de poids
class_weights_tensor = torch.tensor(
    config["class_weights_tensor_order"], dtype=torch.float32
).to(device)

focal_alpha_tensor = torch.tensor(
    config["focal_alpha_tensor_order"], dtype=torch.float32
).to(device)

print(f"Stratégie : {'Focal Loss (γ={})'.format(FOCAL_GAMMA) if USE_FOCAL_LOSS else 'Class Weights (CE pondérée)'}")
print(f"Class weights [pos, neg, neu] : {config['class_weights_tensor_order']}")
print(f"Focal alpha   [pos, neg, neu] : {config['focal_alpha_tensor_order']}")

## 6. Trainer personnalisé avec loss pondérée

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# Charger DziriBERT avec une tête de classification 3 classes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=config["num_labels"],
    id2label=config["id2label"],
    label2id=config["label2id"],
)
model.to(device)

print(f"Modèle : {MODEL_NAME}")
print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")
print(f"Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
class WeightedTrainer(Trainer):
    """
    Trainer HuggingFace avec support pour :
    - CrossEntropy pondérée (class weights)
    - Focal Loss (optionnel)
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if USE_FOCAL_LOSS:
            loss_fn = FocalLoss(
                alpha=focal_alpha_tensor,
                gamma=FOCAL_GAMMA,
            )
            loss = loss_fn(logits, labels)
        else:
            loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
            loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

## 7. Métriques d'évaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)


def compute_metrics(eval_pred):
    """Calcule F1 macro, accuracy, et métriques par classe."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        # F1 par classe
        "f1_positive": f1_score(labels, preds, average=None)[0],
        "f1_negative": f1_score(labels, preds, average=None)[1],
        "f1_neutral": f1_score(labels, preds, average=None)[2],
    }

## 8. Hyper-paramètres & EntraînementHyper-paramètres calibrés pour :- Dataset petit (~1200 exemples train)- GPU Colab (T4 / L4 / A100)- DziriBERT (110M params)

In [ ]:
# ── HYPER-PARAMÈTRES ──
# Ajuster selon votre GPU et vos besoins

EPOCHS = 10                   # Max epochs (early stopping coupe avant)
BATCH_SIZE = 16               # 16 pour T4, 32 pour A100
LEARNING_RATE = 2e-5          # Standard BERT fine-tuning
WARMUP_RATIO = 0.1            # 10% warmup
WEIGHT_DECAY = 0.01           # Régularisation L2
PATIENCE = 3                  # Early stopping patience (epochs sans amélioration)
FP16 = torch.cuda.is_available()  # Mixed precision si GPU

training_args = TrainingArguments(
    output_dir="/content/dziribert-sentiment",
    overwrite_output_dir=True,

    # Entraînement
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=FP16,
    gradient_accumulation_steps=1,

    # Évaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Sauvegarde
    save_total_limit=3,
    report_to="tensorboard",
    logging_dir="/content/logs",

    # Reproductibilité
    seed=42,
    data_seed=42,
)

print("Training arguments configurés :")
print(f"  Epochs      : {EPOCHS} (avec early stopping, patience={PATIENCE})")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  FP16        : {FP16}")
print(f"  Best metric : f1_macro")

## 9. Lancer le fine-tuning

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

# Lancer l'entraînement
print("Début du fine-tuning...")
train_result = trainer.train()

# Résumé
print("\n" + "=" * 50)
print("ENTRAÎNEMENT TERMINÉ")
print("=" * 50)
print(f"  Temps total     : {train_result.metrics['train_runtime']:.0f}s")
print(f"  Loss finale     : {train_result.metrics['train_loss']:.4f}")
print(f"  Epochs réalisés : {train_result.metrics.get('epoch', '?')}")

## 10. Évaluation sur le test set

In [ ]:
# Évaluer sur le test set
test_results = trainer.evaluate(tokenized["test"])

print("=" * 50)
print("RÉSULTATS SUR LE TEST SET")
print("=" * 50)
for k, v in sorted(test_results.items()):
    if k.startswith("eval_"):
        name = k.replace("eval_", "")
        print(f"  {name:20s} : {v:.4f}")

# Seuils MVP
f1_macro = test_results["eval_f1_macro"]
f1_pos = test_results["eval_f1_positive"]
f1_neg = test_results["eval_f1_negative"]
f1_neu = test_results["eval_f1_neutral"]

print("\n" + "=" * 50)
print("VALIDATION DES SEUILS MVP")
print("=" * 50)
print(f"  F1 macro ≥ 0.70 : {'✅ PASS' if f1_macro >= 0.70 else '❌ FAIL'} ({f1_macro:.4f})")
print(f"  F1 positive ≥ 0.60 : {'✅ PASS' if f1_pos >= 0.60 else '❌ FAIL'} ({f1_pos:.4f})")
print(f"  F1 negative ≥ 0.60 : {'✅ PASS' if f1_neg >= 0.60 else '❌ FAIL'} ({f1_neg:.4f})")
print(f"  F1 neutral  ≥ 0.60 : {'✅ PASS' if f1_neu >= 0.60 else '❌ FAIL'} ({f1_neu:.4f})")

In [ ]:
# Classification report détaillé + matrice de confusion
test_preds = trainer.predict(tokenized["test"])
y_pred = np.argmax(test_preds.predictions, axis=-1)
y_true = test_preds.label_ids

label_names = [config["id2label"][str(i)] for i in range(config["num_labels"])]

print("\nClassification Report :")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

print("\nMatrice de confusion :")
print("=" * 50)
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=label_names, columns=[f"pred_{l}" for l in label_names])
print(cm_df.to_string())

## 11. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion heatmap
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_names, yticklabels=label_names,
    ax=axes[0]
)
axes[0].set_xlabel("Prédit")
axes[0].set_ylabel("Réel")
axes[0].set_title("Matrice de confusion — Test set")

# F1 par classe
f1_scores = [f1_pos, f1_neg, f1_neu]
colors = ["#2ecc71", "#e74c3c", "#95a5a6"]
bars = axes[1].bar(label_names, f1_scores, color=colors, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=0.60, color="orange", linestyle="--", label="Seuil MVP (0.60)")
axes[1].axhline(y=0.70, color="red", linestyle="--", label="Seuil cible (0.70)")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("F1 par classe")
axes[1].set_ylim(0, 1)
axes[1].legend()

# Ajouter les valeurs sur les barres
for bar, score in zip(bars, f1_scores):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        f"{score:.3f}", ha="center", va="bottom", fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/content/evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Sauvegarder le modèle

In [ ]:
# Sauvegarder le meilleur modèle
SAVE_DIR = "/content/dziribert-sentiment-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Sauvegarder les résultats de test
results_to_save = {
    "model": MODEL_NAME,
    "strategy": "focal_loss" if USE_FOCAL_LOSS else "class_weights",
    "test_metrics": {k.replace("eval_", ""): round(v, 4) for k, v in test_results.items()},
    "training_config": config,
    "hyperparameters": {
        "epochs_max": EPOCHS,
        "epochs_actual": train_result.metrics.get("epoch", EPOCHS),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "early_stopping_patience": PATIENCE,
        "fp16": FP16,
        "focal_gamma": FOCAL_GAMMA if USE_FOCAL_LOSS else None,
    },
}

with open(f"{SAVE_DIR}/training_results.json", "w") as f:
    json.dump(results_to_save, f, indent=2, ensure_ascii=False)

print(f"Modèle sauvegardé dans : {SAVE_DIR}/")
!ls -la {SAVE_DIR}/

## 13. Exporter le modèleDeux options :- **Option A** : Télécharger en local (zip)- **Option B** : Pousser vers HuggingFace Hub

In [ ]:
# Option A : Télécharger en zip
import shutil
shutil.make_archive("/content/dziribert-sentiment-final", "zip", SAVE_DIR)

from google.colab import files
files.download("/content/dziribert-sentiment-final.zip")

In [ ]:
# Option B : Push vers HuggingFace Hub (décommenter pour utiliser)
# from huggingface_hub import login
# login()  # Entrer votre token HF
#
# model.push_to_hub("rayantr06/dziribert-sentiment-derja")
# tokenizer.push_to_hub("rayantr06/dziribert-sentiment-derja")
# print("Modèle publié sur HuggingFace Hub !")

## 14. Test rapide d'inférence

In [ ]:
from transformers import pipeline

# Charger le modèle fine-tuné
clf = pipeline(
    "text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

# Exemples de test en Derja
test_examples = [
    "رامي عصير ممتاز والله نحبو بزاف",          # positif attendu
    "المنتوج هذا ماشي مليح خلاص عاودت شريتو",    # négatif attendu
    "شحال الثمن تع رامي الجديد؟",                 # neutre attendu
    "Ramy jus top qualité mashallah",              # positif attendu (Arabizi)
    "had el produit dégoûtant wallah",              # négatif attendu (code-switch)
    "win nlgah ramy orange f béjaia?",              # neutre attendu (Arabizi)
]

print("=" * 60)
print("TEST D'INFÉRENCE — DziriBERT Sentiment")
print("=" * 60)
for text in test_examples:
    result = clf(text)[0]
    label = result["label"]
    score = result["score"]
    emoji = {"positive": "😊", "negative": "😠", "neutral": "😐"}.get(label, "❓")
    print(f"  {emoji} [{label:8s} {score:.3f}] {text[:60]}")

## 15. TensorBoard (optionnel)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/logs